# Phase 1: Exploratory Data Analysis & Descriptive Analytics

This notebook loads the local sample Parquet dataset (`data/sample/binance_sample.parquet`), performs descriptive profiling across unique symbols, and generates **5 key visualizations** to understand cryptocurrency market behaviors.

### Architectural & Preprocessing Decisions:
1. **Polars vs. Pandas**: Polars is used for core data processing due to its speed (written in Rust) and lazy evaluation engine. It handles millions of observations without triggering memory leaks or out-of-memory (OOM) errors.
2. **Data Normalization**: Raw Binance datasets contain high-frequency prices in nominal USD levels. We convert them into relative log returns to make the data stationary.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join("..")))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import src.config as config

# Load local sample data
print(f"Loading data from: {config.SAMPLE_PARQUET_PATH}")
df = pl.read_parquet(config.ACTIVE_DATA_PATH)
print(f"Loaded {len(df):,} rows.")

## 1. Descriptive Statistics & Summary Metrics

In [ ]:
# Aggregate key descriptive stats by symbol
df_summary = df.group_by("symbol").agg([
    pl.len().alias("total_observations"),
    pl.col("open_time").min().alias("start_date"),
    pl.col("open_time").max().alias("end_date"),
    pl.col("close").mean().alias("mean_price"),
    pl.col("close").std().alias("std_price"),
    pl.col("volume").mean().alias("mean_volume")
]).sort("total_observations", descending=True)

print("Top 5 most observed symbols in the sample:")
df_summary.head(5).to_pandas()

## 2. Vis 1: Normalized Price Trends
We overlay normalized close price series for the top 5 assets to inspect long-term trend correlations.

In [ ]:
top_5_symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT", "XRPUSDT"]

plt.figure(figsize=(10, 5))
for symbol in top_5_symbols:
    sym_df = df.filter(pl.col("symbol") == symbol).sort("open_time").to_pandas()
    # Scale close price to start at 100 for relative comparison
    sym_df["scaled_close"] = (sym_df["close"] / sym_df["close"].iloc[0]) * 100
    plt.plot(sym_df["open_time"], sym_df["scaled_close"], label=symbol, alpha=0.85)

plt.title("Normalized Close Price Trends (Indexed to 100)")
plt.xlabel("Timestamp (UTC)")
plt.ylabel("Normalized Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Vis 2: Return Distributions
We inspect the log return distribution of the configured target symbol to evaluate normality (skewness/kurtosis).

In [ ]:
target_df = df.filter(pl.col("symbol") == config.TARGET_SYMBOL).sort("open_time").to_pandas()
target_df["log_return"] = np.log(target_df["close"] / target_df["close"].shift(1))

plt.figure(figsize=(8, 4.5))
sns.histplot(target_df["log_return"].dropna(), bins=100, kde=True, color="indigo")
plt.title(f"{config.TARGET_SYMBOL} Log Return Distribution (1m intervals)")
plt.xlabel("Log Return")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Skewness: {target_df['log_return'].skew():.4f}")
print(f"Kurtosis (excess): {target_df['log_return'].kurtosis():.4f}")

## 4. Vis 3: Volatility Analysis
We calculate rolling standard deviation of log returns (30-period window) to inspect volatility clustering regimes.

In [ ]:
target_df["rolling_vol"] = target_df["log_return"].rolling(window=30).std()

plt.figure(figsize=(10, 4.5))
plt.plot(target_df["open_time"], target_df["rolling_vol"], color="crimson", alpha=0.8)
plt.title(f"{config.TARGET_SYMBOL} 30-Minute Rolling Volatility")
plt.xlabel("Timestamp")
plt.ylabel("Standard Deviation")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Vis 4: Average Volume by Hour of Day
We analyze seasonal trading activity based on the hour of day.

In [ ]:
target_df["hour"] = pd.to_datetime(target_df["open_time"]).dt.hour
hourly_volume = target_df.groupby("hour")["volume"].mean()

plt.figure(figsize=(8, 4))
plt.bar(hourly_volume.index, hourly_volume.values, color="seagreen", alpha=0.85)
plt.title(f"{config.TARGET_SYMBOL} Average Volume by Hour of Day (UTC)")
plt.xlabel("Hour of Day")
plt.ylabel("Average Transaction Volume")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Vis 5: Asset Return Correlation Heatmap
We calculate return correlations across top cryptocurrency assets to understand system co-movements.

In [ ]:
returns_dict = {}
for symbol in top_5_symbols:
    sym_df = df.filter(pl.col("symbol") == symbol).sort("open_time").to_pandas()
    sym_df["log_return"] = np.log(sym_df["close"] / sym_df["close"].shift(1))
    returns_dict[symbol] = sym_df.set_index("open_time")["log_return"]

corr_df = pd.DataFrame(returns_dict).dropna()
plt.figure(figsize=(6, 5))
sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm", fmt=".3f", vmin=-1, vmax=1)
plt.title("Cryptocurrency Log Return Correlation Heatmap")
plt.tight_layout()
plt.show()

## 7. Feature Engineering & Preprocessing for EDA
To perform basic and advanced exploratory data analysis on the models' features, we first compute the technical indicators and stationary features using our source package logic. We focus on the configured target trading pair (`config.TARGET_SYMBOL`).

In [ ]:
# Import feature engineering functions
from src.features.indicators import compute_indicators, compute_stationary_features

# Filter data for target symbol and compute features
df_target = df.filter(pl.col("symbol") == config.TARGET_SYMBOL)
df_indicators = compute_indicators(df_target)
df_features = compute_stationary_features(df_indicators)

# Drop initial nulls resulting from rolling windows
df_features = df_features.drop_nulls()

# Define the list of engineered features used for model training
feature_cols = [
    "close_to_sma_15", "close_to_sma_50",
    "close_to_ema_15", "close_to_ema_50",
    "bb_position",
    "macd_line_norm", "macd_signal_norm", "macd_hist_norm",
    "volatility_30", "rsi_14", "log_return"
]

print(f"Computed stationary features for {config.TARGET_SYMBOL}. Final clean rows: {len(df_features):,}")

## 8. Vis 6: Distributions of Stationary Features (Basic EDA)
We compute key descriptive statistics (mean, median, standard deviation, skewness, and kurtosis) and visualize the probability density function / histogram of each feature to inspect normality and shape.

In [ ]:
# Calculate descriptive stats for each feature
stats_list = []
for col in feature_cols:
    series = df_features[col].to_numpy()
    mean = float(np.mean(series))
    median = float(np.median(series))
    std = float(np.std(series))
    skew = float(pd.Series(series).skew())
    kurt = float(pd.Series(series).kurtosis())

    stats_list.append({
        "Feature": col,
        "Mean": mean,
        "Median": median,
        "Std": std,
        "Skewness": skew,
        "Kurtosis": kurt
    })

df_stats = pd.DataFrame(stats_list)
print("Feature Summary Statistics:")
df_stats.set_index("Feature")

In [ ]:
# Plot distributions for all features
plt.figure(figsize=(15, 12))
for i, col in enumerate(feature_cols, 1):
    plt.subplot(4, 3, i)
    sns.histplot(df_features[col].to_numpy(), kde=True, bins=50, color="teal")
    plt.title(f"Distribution of {col}")
    plt.xlabel("")
plt.tight_layout()
plt.show()

### Interpretation of Feature Distributions:
1. **Fat Tails (Leptokurtosis)**: Features like `log_return` (Kurtosis ~ 81.9) and moving average deviations (`close_to_ema_15` Kurtosis ~ 63.4, `close_to_sma_15` Kurtosis ~ 58.9) exhibit extremely high kurtosis. This is typical of financial returns and deviations, representing heavy tails where extreme moves are far more common than in a standard normal distribution.
2. **Asymmetric Skewness**:
   - `volatility_30` is highly right-skewed (Skewness ~ 4.4), reflecting long quiet periods with low volatility interrupted by occasional massive volatility spikes (clustering).
   - Moving average deviations and returns are negatively skewed (~ -0.6 to -1.0), meaning there is a slight left tail asymmetry where large downward price drops occur faster and more violently than large upward moves.
3. **Platykurtic Bounded Features**:
   - `bb_position` has a kurtosis of -0.81 and skewness of -0.01. This is close to a flat uniform distribution, showing that the price oscillates widely within the Bollinger Bands, spending significant time near both the upper and lower limits.
   - `rsi_14` has near-zero skewness (0.02) and low kurtosis (-0.20), representing a bounded distribution centered around 50.

## 9. Vis 7: Outlier Profiling (Basic EDA)
We check for outliers in each feature using the Interquartile Range (IQR) method. An outlier is defined as any value that lies outside of the range $[Q1 - 1.5 \times \text{IQR}, Q3 + 1.5 \times \text{IQR}]$.

In [ ]:
outliers_list = []
for col in feature_cols:
    series = df_features[col].to_numpy()
    q25 = float(np.percentile(series, 25))
    q75 = float(np.percentile(series, 75))
    iqr = q75 - q25
    lower_bound = q25 - 1.5 * iqr
    upper_bound = q75 + 1.5 * iqr

    outliers_mask = (series < lower_bound) | (series > upper_bound)
    num_outliers = int(np.sum(outliers_mask))
    pct_outliers = (num_outliers / len(series)) * 100

    outliers_list.append({
        "Feature": col,
        "Q25": q25,
        "Q75": q75,
        "IQR": iqr,
        "Outliers Count": num_outliers,
        "Outliers (%)": f"{pct_outliers:.2f}%"
    })

df_outliers = pd.DataFrame(outliers_list)
print("Feature Outlier Analysis:")
df_outliers.set_index("Feature")

In [ ]:
# Visualize outliers with boxplots
plt.figure(figsize=(15, 12))
for i, col in enumerate(feature_cols, 1):
    plt.subplot(4, 3, i)
    sns.boxplot(y=df_features[col].to_numpy(), color="salmon")
    plt.title(f"Boxplot of {col}")
    plt.ylabel("")
plt.tight_layout()
plt.show()

### Interpretation of Outliers:
- **No Outlier Removal**: We explicitly choose **NOT** to remove or clip outliers in this dataset. In high-frequency 1m cryptocurrency data, these "outliers" correspond to periods of high volatility, news/liquidation shocks, and rapid price discovery. Removing these points would eliminate the exact periods of market distress and high momentum that are most critical for our ML classifiers to predict.
- **Bounded Features**: Bounded features like `bb_position` (0% outliers) and `rsi_14` (0.66% outliers) are stable and have very few extreme deviations.
- **Unbounded Features**: Unbounded features (e.g. `log_return` with 8.34% outliers, `close_to_ema_15` with 7.61% outliers) show a high percentage of outliers compared to a normal distribution (where IQR only flags 0.7% of points). This further confirms the heavy-tailed nature of financial return series.

## 10. Vis 8: Feature Correlation Heatmap & Multicollinearity (Advanced EDA)
We compute the Pearson correlation coefficient matrix for our features to identify collinear patterns that could affect linear model estimation (Logistic Regression).

In [ ]:
# Calculate correlation matrix
df_pandas = df_features[feature_cols].to_pandas()
corr_matrix = df_pandas.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap (Multicollinearity Check)")
plt.show()

### Interpretation of Feature Correlation:
- **High Multicollinearity**: There are extremely high positive correlations between close-to-moving-average deviations and norm MACD values (e.g., `close_to_sma_15` and `close_to_ema_15` have correlation of 0.99; `macd_line_norm` and `close_to_ema_15` have correlation of 0.93).
- **Model Training Implications**: Linear classifiers (like Logistic Regression) will face multicollinearity, which can destabilize coefficient estimates (large standard errors, oscillating coefficients). A tree-based model (like Random Forest) or a regularized linear model (L2 penalty) should be preferred to handle these highly correlated indicators.
- **Low Correlation features**: `bb_position`, `rsi_14`, `volatility_30`, and `log_return` show weak correlation with other indicators, indicating they provide unique non-redundant information.

## 11. Vis 9: Stationarity Visual Verification (Advanced EDA)
Machine learning models degrade if raw nominal price trends drift. We visualize raw nominal price levels vs. the stationary close price to SMA 15 ratio over a 1000-minute window to show the benefit of stationary transformations.

In [ ]:
# Plot Close vs Close to SMA 15
plot_df = df_features.head(1000).to_pandas()

fig, ax1 = plt.subplots(figsize=(10, 4.5))

color = 'tab:blue'
ax1.set_xlabel('Time (1m intervals)')
ax1.set_ylabel('Raw Close Price (USD)', color=color)
ax1.plot(plot_df['open_time'], plot_df['close'], color=color, label='Close Price')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Close to SMA 15 (Stationary)', color=color)
ax2.plot(plot_df['open_time'], plot_df['close_to_sma_15'], color=color, linestyle='--', label='Close to SMA 15')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Stationarity Comparison: Raw Drifting Price vs. Mean-Reverting Stationary Feature')
fig.tight_layout()
plt.show()

### Interpretation of Stationarity:
- The raw Close Price drifts upward/downward and is non-stationary, meaning its mean and variance change over time.
- The `close_to_sma_15` feature oscillates around a constant mean of 0 with a stable variance. By transforming nominal prices into stationary percentage offsets, we ensure the model learns patterns that generalize across all price regimes (e.g. from $30k BTC to $100k BTC).

## 12. Vis 10: Feature Interaction with Target Variable (Advanced EDA)
We compute the future binary price movement target (direction classification target representing return over the future horizon $N$) and inspect how our features vary between the target classes.

In [ ]:
# Compute target label (Binary movement over N future minutes)
df_labeled = df_features.with_columns([
    (pl.col("close").shift(-config.FUTURE_HORIZON).over("symbol") / pl.col("close") - 1.0)
    .alias("future_return")
]).with_columns([
    pl.when(pl.col("future_return") > config.TARGET_THRESHOLD)
    .then(1)
    .otherwise(0)
    .alias("target")
]).drop_nulls()

df_labeled_pd = df_labeled[feature_cols + ["target", "future_return"]].to_pandas()

# Plot boxplots of features grouped by the binary target label
plot_features = ["close_to_sma_15", "bb_position", "macd_hist_norm", "volatility_30", "rsi_14", "log_return"]
plt.figure(figsize=(14, 9))
for i, col in enumerate(plot_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(x="target", y=col, data=df_labeled_pd, palette="Set2")
    plt.title(f"{col} by Target Class")
    plt.xlabel("Target (0=DOWN/FLAT, 1=UP)")
plt.tight_layout()
plt.show()

In [ ]:
# Compute correlation of each feature with the future return
corrs = df_labeled_pd[feature_cols].corrwith(df_labeled_pd["future_return"])
print("Pearson Correlation of Features with Future Return:")
print(corrs.sort_values(ascending=False))

### Interpretation of Target Interactions:
- **Low Linear Correlation**: The Pearson correlation values of features with the future return are very close to zero (typically $< 0.01$). This is expected in high-frequency markets, indicating that predicting price movements at a 1-minute interval is highly challenging and linear relationships are very weak.
- **Non-Linear Dynamics**: The target class distributions (UP vs. DOWN) overlap heavily in the boxplots. The classifiers will need to utilize non-linear interactions and joint feature combinations to extract predictive signals.